### Imports/Setup

In [ ]:
import os

from dotenv import load_dotenv
from qdrant_client import QdrantClient, models
from qdrant_client.models import Document
import matplotlib.pyplot as plt
from PIL import Image

### Load embedding models and logic

In [40]:
import numpy as np
import torch
from transformers import AutoModel, AutoProcessor

MAX_SIDE = 1024

# load the model and processor
MODEL_ID = "google/siglip2-base-patch16-224"
device = 'cuda' if torch.cuda.is_available() else "cpu"

model = AutoModel.from_pretrained(MODEL_ID).to(device)
processor = AutoProcessor.from_pretrained(MODEL_ID)

def l2_normalize(x: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(x, axis=-1, keepdims=True)
    norms[norms == 0] = 1.0
    return (x / norms).astype('float32')


@torch.no_grad()
def embed_image(path: str):
    """
    Given an image path an image is loaded, preprocessed,
    and embedded using a locally run siglip2 model
    """
    # Load Image from Path
    try:
        image = Image.open(path)
    except OSError as e:
        raise FileNotFoundError(f"Invalid image path: {path}") from e

    # Convert to RGB and downscale if necessary
    image = image.convert("RGB")
    if max(image.size) > MAX_SIDE:
        image.thumbnail((MAX_SIDE, MAX_SIDE), Image.LANCZOS)

    inputs = processor(image, return_tensors="pt").to(device)
    feats = model.get_image_features(**inputs).pooler_output
    vector = feats.cpu().numpy()
    return l2_normalize(vector)

@torch.no_grad()
def embed_text(x: str):
    """
    Given a string, embeds it using
    a locally run siglip2 model
    """
    inputs = processor(text = x, return_tensors = "pt").to(device)
    feats = model.get_text_features(**inputs).pooler_output
    vector = feats.cpu().numpy()
    return l2_normalize(vector)

Loading weights: 100%|██████████| 408/408 [00:00<00:00, 3215.63it/s]


### Connect to Qdrant Cluster

In [41]:
load_dotenv()
client = QdrantClient(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"],
    cloud_inference=True
)
COLLECTION = "RKDTestSet4"

### Method Parsing and Querying Logic

In [42]:
def clean_path(path):
    """Normalizes a pasted Windows path: drops surrounding whitespace/quotes and fixes separators."""
    return os.path.normpath(path.strip().strip('"').strip("'"))

def parse_method(method, text):
    """Returns the (query, using) pair Qdrant needs for a given scoring method."""
    if method == "keyword":
        return Document(text=text, model="Qdrant/bm25"), "title-sparse"
    if method == "conceptual":
        return Document(text=text, model="sentence-transformers/all-MiniLM-L6-v2"), "description"
    if method == "visual":
        return embed_text(text)[0].tolist(), "siglip_description"
    if method == 'image':
        return embed_image(clean_path(text)), 'image'
    else:
        raise ValueError('this method doesnt exist')

def single_method_search(method, text, limit = 6):
    """Runs one scoring method in isolation and returns its ranked points."""
    query, using = parse_method(method, text)
    result = client.query_points(
        collection_name = COLLECTION,
        query = query,
        using = using,
        limit = limit,
        with_payload=True,
    )
    return result.points

def composite_search(methods: list, text, path = None, limit=6):
    """Fuses the given methods with Qdrant's built-in Reciprocal Rank Fusion."""
    prefetch = []
    for method in methods:
        query, using = parse_method(method, text)
        prefetch.append(models.Prefetch(query=query, using = using, limit = limit))

    # If we're using both text and an image path to query
    if path:
        query, using = parse_method('image', path)
        prefetch.append(models.Prefetch(query = query, using = using, limit = limit))

    result = client.query_points(
        collection_name = COLLECTION,
        prefetch = prefetch,
        query = models.FusionQuery(fusion = models.Fusion.RRF),
        with_payload = True,
        limit = limit,
    )
    return result.points

### Single Method Search

In [ ]:
# Valid methods are: keyword, conceptual, visual, and image
METHOD = ''
# If searching via image, paste the path as a raw string: r"C:\...\file.jpg"
INPUT = ''

points = single_method_search(METHOD, INPUT)
for i, point in enumerate(points):
    print(f'{i + 1}. {point.payload['title_en']}, Score: {point.score}, Link: https://rkd.nl/images/{point.id}')

FileNotFoundError: Invalid image path: .

### Hybrid Search

In [ ]:
METHODS = []
INPUT = ""
# If searching with an image as well, paste the path as a raw string: r"C:\...\file.jpg"
PATH = r""

points = composite_search(METHODS, INPUT, PATH)
for i, point in enumerate(points):
    print(f'{i + 1}. {point.payload['title_en']}, Score: {point.score}, Link: https://rkd.nl/images/{point.id}')

1. Boys playing, Score: 0.5, Link: https://rkd.nl/images/346288
2. None, Score: 0.5, Link: https://rkd.nl/images/69701
3. Henry IV entrusts Maria de' Medici with the regency over France, 20 March 1610, Score: 0.5, Link: https://rkd.nl/images/261031
4. Study of a shed, Score: 0.33333334, Link: https://rkd.nl/images/34059
5. Poor folk drinking in a tavern, Score: 0.33333334, Link: https://rkd.nl/images/25815
6. Saint Aloysius Gonzaga in ecstasy, Score: 0.33333334, Link: https://rkd.nl/images/236060
